# Final Modeling Loyalty Strategy

This notebook translates the EDA work into operational scoring outputs. It trains the two propensity models, evaluates them on the latest four reference months, checks the model-family tradeoff, and then converts the scores into CRM segments.

The business goal is to support a targeted marketing campaign by delivering the right message to the right user at the right time.

The modeling setup stays intentionally simple. The objective is not maximum algorithmic complexity, but a scoring framework that is stable, interpretable, and easy to connect to downstream CRM actions.

First, we run `eda.ipynb` so the shared snapshot artifacts are written to `artifacts/eda_training_artifacts.pkl`.

In [17]:
from web.engine import CRMDecisionEngine
from pathlib import Path
from typing import Any, Callable, cast

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots
from sklearn.base import BaseEstimator
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import accuracy_score, average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils import shuffle
from sklearn.utils.class_weight import compute_class_weight

PROJECT_ROOT = Path('.')
EDA_ARTIFACT_PATH = PROJECT_ROOT / 'artifacts' / 'eda_training_artifacts.pkl'
OUTPUT_DIR = PROJECT_ROOT / 'artifacts' / 'final'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_LABEL_MAP = {
    'churn': 'Churn',
    're_engage': 'Re-engagement',
}
MODEL_FAMILY_LABELS = {
    'logistic_regression': 'Logistic Regression',
    'random_forest': 'Random Forest',
}
PRESENTATION_COLORS = {
    'blue': '#2563eb',
    'blue_fill': 'rgba(37, 99, 235, 0.16)',
    'blue_light': '#bfdbfe',
    'orange': '#ea580c',
    'orange_fill': 'rgba(234, 88, 12, 0.14)',
    'green': '#059669',
    'green_fill': 'rgba(5, 150, 105, 0.14)',
    'red': '#dc2626',
    'slate': '#475569',
    'ink': '#111827',
    'grid': '#e2e8f0',
}
FIGURE_DIMENSIONS = {
    'validation_history': {'width': 1150, 'height': 420},
    'model_family_check': {'width': 1150, 'height': 440},
    'calibration_distribution': {'width': 1150, 'height': 480},
    'crm_segment_distribution': {'width': 1320, 'height': 460},
}


def apply_presentation_layout(
    fig: go.Figure,
    *,
    xaxis_title: str | None = None,
    yaxis_title: str | None = None,
    width: int | None = None,
    height: int | None = None,
) -> go.Figure:
    fig.update_layout(
        template='plotly_white',
        paper_bgcolor='white',
        plot_bgcolor='white',
        font={'family': 'Arial', 'size': 15, 'color': '#334155'},
        margin={'l': 80, 'r': 30, 't': 28, 'b': 70},
        width=width,
        height=height,
    )
    if xaxis_title is not None:
        fig.update_xaxes(title=xaxis_title, showgrid=True,
                         gridcolor=PRESENTATION_COLORS['grid'])
    if yaxis_title is not None:
        fig.update_yaxes(title=yaxis_title, showgrid=True,
                         gridcolor=PRESENTATION_COLORS['grid'])
    return fig



if not EDA_ARTIFACT_PATH.exists():
    raise FileNotFoundError(
        f'Missing {
            EDA_ARTIFACT_PATH}. Run eda.ipynb first to materialize the shared training artifacts.'
    )

In [18]:
artifacts = pd.read_pickle(EDA_ARTIFACT_PATH)
snapshot_features = artifacts['snapshot_features'].copy()
snapshot_labels = artifacts['snapshot_labels'].copy()
model_frames = artifacts['model_frames']

snapshots = snapshot_features.merge(
    snapshot_labels,
    on=['idSSO', 'reference_date'],
    how='inner',
)
snapshots['reference_date'] = pd.to_datetime(snapshots['reference_date'])
snapshots['reference_month'] = snapshots['reference_date'].dt.strftime('%Y-%m')
snapshots['points_gap_proxy'] = (
    snapshots['reward_threshold_points'] - snapshots['totalPoints'].fillna(0)
).clip(lower=0)

churn_frame = model_frames['churn'].copy()
churn_frame['reference_date'] = pd.to_datetime(churn_frame['reference_date'])
churn_frame['reference_month'] = churn_frame['reference_date'].dt.strftime(
    '%Y-%m')
churn_frame['points_gap_proxy'] = (
    churn_frame['reward_threshold_points'] -
    churn_frame['totalPoints'].fillna(0)
).clip(lower=0)
churn_frame['churn_30_to_60'] = churn_frame['churn_30_to_60'].astype(int)

engagement_frame = model_frames['engagement'].copy()
engagement_frame['reference_date'] = pd.to_datetime(
    engagement_frame['reference_date'])
engagement_frame['reference_month'] = engagement_frame['reference_date'].dt.strftime(
    '%Y-%m')
engagement_frame['points_gap_proxy'] = (
    engagement_frame['reward_threshold_points'] -
    engagement_frame['totalPoints'].fillna(0)
).clip(lower=0)
engagement_frame['re_engage_30d'] = engagement_frame['re_engage_30d'].astype(
    int)

shared_features = artifacts['feature_columns'] + \
    ['points_gap_proxy', 'reference_month']
categorical_features = ['child_age_bucket', 'platform', 'reference_month']

print('Churn rows:', len(churn_frame))
print('Re-engagement rows:', len(engagement_frame))
print('Reference dates:', snapshots['reference_date'].min(
).date(), '->', snapshots['reference_date'].max().date())

Churn rows: 5787
Re-engagement rows: 64924
Reference dates: 2024-04-01 -> 2025-08-01


## Modeling Approach

Both propensities use the same stack: shared snapshot features, a temporal holdout on the most recent four reference months, and a class-weighted logistic regression with basic preprocessing.

Validation is done with a temporal split rather than a random row split. The models are trained on earlier months and evaluated on the latest ones, which is closer to the real CRM setting where patterns learned from the past are used to score future periods. In this notebook, that setup gives 5,787 churn rows and 64,924 re-engagement rows, with reference dates from Apr 2024 to Aug 2025.

The feature set remains compact: 41 base features from `eda.ipynb`, plus `points_gap_proxy` and `reference_month`, for 43 candidates in total. Numeric fields use median imputation and scaling, categorical fields use a `missing` bucket plus one-hot encoding, and the logistic model uses `class_weight='balanced'`.

Median imputation fills missing numeric values with the median observed in the training data, which is a simple way to avoid dropping rows while limiting the effect of extreme values. Scaling then puts numeric features on a comparable scale so that variables measured in very different units do not dominate the logistic model. One-hot encoding converts each categorical value into a separate binary column, which allows the model to treat categories such as platform or reference month as structured inputs rather than raw text labels.

The next sections evaluate how well this setup performs on the holdout period. They also compare the logistic baseline with a random forest to check whether additional model flexibility materially changes the result.


In [19]:
def make_deciles(probabilities: pd.Series) -> pd.Series:
    ranked = probabilities.rank(method='first')
    deciles = pd.qcut(ranked, 10, labels=range(1, 11), duplicates='drop')
    return pd.Series(deciles, index=probabilities.index, dtype='Int64')


def split_by_time(frame: pd.DataFrame, holdout_months: int = 4) -> tuple[pd.DataFrame, pd.DataFrame]:
    reference_dates = sorted(frame['reference_date'].dropna().unique())
    if len(reference_dates) <= holdout_months:
        raise ValueError(
            'Not enough reference dates for the requested holdout split.')
    valid_dates = set(reference_dates[-holdout_months:])
    train_df = cast(
        pd.DataFrame, frame.loc[~frame['reference_date'].isin(valid_dates)].copy())
    valid_df = cast(
        pd.DataFrame, frame.loc[frame['reference_date'].isin(valid_dates)].copy())
    return cast(tuple[pd.DataFrame, pd.DataFrame], (train_df, valid_df))


def build_preprocess(
    feature_columns: list[str],
    categorical_columns: list[str],
    *,
    scale_numeric: bool,
) -> ColumnTransformer:
    usable_categoricals = [
        col for col in categorical_columns if col in feature_columns]
    numeric_columns = [
        col for col in feature_columns if col not in usable_categoricals]
    numeric_steps: list[tuple[str, BaseEstimator]] = [
        ('imputer', SimpleImputer(strategy='median'))]
    if scale_numeric:
        numeric_steps.append(('scaler', StandardScaler()))
    return ColumnTransformer(
        transformers=[
            ('numeric', Pipeline(steps=numeric_steps), numeric_columns),
            (
                'categorical',
                Pipeline(
                    steps=[
                        ('imputer', SimpleImputer(
                            strategy='constant', fill_value='missing')),
                        ('onehot', OneHotEncoder(handle_unknown='ignore')),
                    ]
                ),
                usable_categoricals,
            ),
        ]
    )


def build_logistic_pipeline(feature_columns: list[str], categorical_columns: list[str]) -> Pipeline:
    preprocess = build_preprocess(
        feature_columns, categorical_columns, scale_numeric=True)
    return Pipeline(
        steps=[
            ('preprocess', preprocess),
            (
                'model',
                LogisticRegression(
                    class_weight='balanced',
                    max_iter=1000,
                    random_state=42,
                    solver='liblinear',
                ),
            ),
        ]
    )


def build_random_forest_pipeline(feature_columns: list[str], categorical_columns: list[str]) -> Pipeline:
    preprocess = build_preprocess(
        feature_columns, categorical_columns, scale_numeric=False)
    return Pipeline(
        steps=[
            ('preprocess', preprocess),
            (
                'model',
                RandomForestClassifier(
                    n_estimators=400,
                    max_depth=10,
                    min_samples_leaf=20,
                    class_weight='balanced_subsample',
                    n_jobs=-1,
                    random_state=42,
                ),
            ),
        ]
    )


def clean_feature_name(feature_name: str) -> str:
    return feature_name.replace('numeric__', '').replace('categorical__', '')


def prepare_model_frame(
    frame: pd.DataFrame,
    *,
    feature_columns: list[str],
    categorical_columns: list[str],
) -> tuple[pd.DataFrame, list[str]]:
    usable_features = [
        col for col in feature_columns if col in frame.columns and not cast(pd.Series, frame[col]).dropna().empty
    ]
    model_frame = frame.copy()
    for column in usable_features:
        if column in categorical_columns:
            model_frame[column] = model_frame[column].astype(
                'string').fillna('missing').astype(str)
        else:
            model_frame[column] = pd.to_numeric(
                model_frame[column], errors='coerce').astype(float)
    return model_frame, usable_features


def extract_feature_relevance(model_pipeline: Pipeline, *, top_n: int = 15) -> pd.DataFrame:
    feature_names = [
        clean_feature_name(name)
        for name in model_pipeline.named_steps['preprocess'].get_feature_names_out()
    ]
    model_step = model_pipeline.named_steps['model']
    if hasattr(model_step, 'coef_'):
        feature_relevance = pd.DataFrame(
            {'feature': feature_names, 'relevance': model_step.coef_[0]})
        feature_relevance['abs_relevance'] = feature_relevance['relevance'].abs()
        feature_relevance['direction'] = np.where(
            feature_relevance['relevance'] >= 0,
            'raises score',
            'lowers score',
        )
        feature_relevance['relevance_type'] = 'coefficient'
    elif hasattr(model_step, 'feature_importances_'):
        feature_relevance = pd.DataFrame(
            {'feature': feature_names, 'relevance': model_step.feature_importances_})
        feature_relevance['abs_relevance'] = feature_relevance['relevance']
        feature_relevance['direction'] = 'non-directional'
        feature_relevance['relevance_type'] = 'feature_importance'
    else:
        return pd.DataFrame(columns=['feature', 'relevance', 'abs_relevance', 'direction', 'relevance_type'])
    return feature_relevance.sort_values('abs_relevance', ascending=False).head(top_n)


def calibration_table(frame: pd.DataFrame, probability_column: str, label_column: str) -> pd.DataFrame:
    scored = frame.copy()
    scored['score_decile'] = make_deciles(scored[probability_column])
    return (
        scored.groupby('score_decile', dropna=False)
        .agg(rows=(label_column, 'size'), avg_prediction=(probability_column, 'mean'), actual_rate=(label_column, 'mean'))
        .reset_index()
        .sort_values('score_decile')
    )


def evaluate_predictions(frame: pd.DataFrame, probability_column: str, label_column: str) -> pd.Series:
    y_true = frame[label_column].astype(int)
    y_prob = frame[probability_column].clip(1e-6, 1 - 1e-6)
    top_n = max(1, int(np.ceil(len(frame) * 0.10)))
    top_rate = float(frame.nlargest(top_n, probability_column)
                     [label_column].mean())
    return pd.Series(
        {
            'rows': int(len(frame)),
            'positive_rate': float(y_true.mean()),
            'roc_auc': float(roc_auc_score(y_true, y_prob)),
            'average_precision': float(average_precision_score(y_true, y_prob)),
            'brier_score': float(brier_score_loss(y_true, y_prob)),
            'log_loss': float(log_loss(y_true, y_prob)),
            'top_decile_precision': top_rate,
        }
    )


def fit_propensity_model(
    frame: pd.DataFrame,
    *,
    label_column: str,
    score_column: str,
    feature_columns: list[str],
    categorical_columns: list[str],
    pipeline_builder: Callable[[list[str], list[str]],
                               Pipeline] = build_logistic_pipeline,
    model_family: str = 'logistic_regression',
) -> dict[str, Any]:
    model_frame, usable_features = prepare_model_frame(
        frame,
        feature_columns=feature_columns,
        categorical_columns=categorical_columns,
    )

    train_df, valid_df = split_by_time(model_frame)
    validation_model = pipeline_builder(usable_features, categorical_columns)
    validation_model.fit(train_df[usable_features],
                         train_df[label_column].astype(int))

    valid_scored = valid_df.copy()
    valid_scored[score_column] = validation_model.predict_proba(
        valid_df[usable_features])[:, 1]

    final_model = pipeline_builder(usable_features, categorical_columns)
    final_model.fit(model_frame[usable_features],
                    model_frame[label_column].astype(int))

    full_scored = model_frame.copy()
    full_scored[score_column] = final_model.predict_proba(
        model_frame[usable_features])[:, 1]

    feature_relevance = extract_feature_relevance(final_model)
    top_coefficients = pd.DataFrame(
        columns=['feature', 'coefficient', 'abs_coefficient'])
    if not feature_relevance.empty and feature_relevance['relevance_type'].iloc[0] == 'coefficient':
        top_coefficients = feature_relevance.rename(
            columns={'relevance': 'coefficient',
                     'abs_relevance': 'abs_coefficient'}
        )[['feature', 'coefficient', 'abs_coefficient']]

    return {
        'validation_model': validation_model,
        'final_model': final_model,
        'features': usable_features,
        'validation_scores': valid_scored,
        'full_scores': full_scored,
        'validation_metrics': evaluate_predictions(valid_scored, score_column, label_column),
        'calibration': calibration_table(valid_scored, score_column, label_column),
        'feature_relevance': feature_relevance,
        'top_coefficients': top_coefficients,
        'model_family': model_family,
    }


def build_epoch_metric_history(
    frame: pd.DataFrame,
    *,
    label_column: str,
    feature_columns: list[str],
    categorical_columns: list[str],
    target_label: str,
    epochs: int = 24,
) -> pd.DataFrame:
    model_frame, usable_features = prepare_model_frame(
        frame,
        feature_columns=feature_columns,
        categorical_columns=categorical_columns,
    )
    train_df, valid_df = split_by_time(model_frame)

    preprocess = build_preprocess(
        usable_features, categorical_columns, scale_numeric=True)
    X_train = preprocess.fit_transform(train_df[usable_features])
    X_valid = preprocess.transform(valid_df[usable_features])

    y_train = train_df[label_column].astype(int).to_numpy()
    y_valid = valid_df[label_column].astype(int).to_numpy()

    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.array([0, 1]),
        y=y_train,
    )
    classifier = SGDClassifier(
        loss='log_loss',
        penalty='l2',
        alpha=0.0005,
        class_weight={0: float(class_weights[0]), 1: float(class_weights[1])},
        random_state=42,
    )
    rng = np.random.default_rng(42)
    history: list[dict[str, object]] = []

    for epoch in range(1, epochs + 1):
        shuffled_epoch = cast(
            tuple[Any, Any],
            shuffle(
                X_train,
                y_train,
                random_state=int(rng.integers(0, 1_000_000)),
            ),
        )
        X_epoch, y_epoch = shuffled_epoch
        if epoch == 1:
            classifier.partial_fit(X_epoch, y_epoch, classes=np.array([0, 1]))
        else:
            classifier.partial_fit(X_epoch, y_epoch)

        train_probabilities = classifier.predict_proba(X_train)[:, 1]
        valid_probabilities = classifier.predict_proba(X_valid)[:, 1]

        history.extend(
            [
                {
                    'target': target_label,
                    'epoch': epoch,
                    'split': 'train',
                    'accuracy': float(accuracy_score(y_train, classifier.predict(X_train))),
                    'roc_auc': float(roc_auc_score(y_train, train_probabilities)),
                },
                {
                    'target': target_label,
                    'epoch': epoch,
                    'split': 'validation',
                    'accuracy': float(accuracy_score(y_valid, classifier.predict(X_valid))),
                    'roc_auc': float(roc_auc_score(y_valid, valid_probabilities)),
                },
            ]
        )

    return pd.DataFrame(history)


def select_epoch_history(epoch_history: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    accuracy_table = (
        epoch_history.pivot(index=['target', 'epoch'],
                            columns='split', values='accuracy')
        .reset_index()
        .rename(columns={'train': 'train_accuracy', 'validation': 'validation_accuracy'})
    )
    roc_auc_table = (
        epoch_history.pivot(index=['target', 'epoch'],
                            columns='split', values='roc_auc')
        .reset_index()
        .rename(columns={'train': 'train_roc_auc', 'validation': 'validation_roc_auc'})
    )
    epoch_table = (
        accuracy_table
        .merge(roc_auc_table, on=['target', 'epoch'], how='inner')
        .sort_values(['target', 'epoch'])
        .reset_index(drop=True)
    )
    epoch_table['chosen_epoch'] = False

    chosen_rows: list[pd.Series] = []
    for target_key, target_slice in epoch_table.groupby('target', sort=False):
        chosen_row = (
            target_slice.sort_values(
                ['validation_accuracy', 'validation_roc_auc', 'epoch'],
                ascending=[False, False, True],
            )
            .iloc[0]
        )
        chosen_rows.append(chosen_row)
        epoch_table.loc[
            (epoch_table['target'] == target_key) & (
                epoch_table['epoch'] == int(chosen_row['epoch'])),
            'chosen_epoch',
        ] = True

    chosen_epoch_summary = pd.DataFrame(chosen_rows).reset_index(drop=True)
    chosen_epoch_summary['target_label'] = chosen_epoch_summary['target'].map(
        MODEL_LABEL_MAP)
    chosen_epoch_summary = chosen_epoch_summary[[
        'target',
        'target_label',
        'epoch',
        'train_accuracy',
        'validation_accuracy',
        'train_roc_auc',
        'validation_roc_auc',
    ]].rename(columns={'epoch': 'chosen_epoch'})

    return epoch_table, chosen_epoch_summary



def add_epoch_metric_traces(
    fig: go.Figure,
    target_history: pd.DataFrame,
    *,
    chosen_epoch: int,
    metric_column: str,
    metric_label: str,
    row: int,
    col: int,
    show_legend: bool,
) -> None:
    split_labels = {
        'train': f'Train {metric_label}',
        'validation': f'Validation {metric_label}',
    }
    split_colors = {
        'train': PRESENTATION_COLORS['blue'],
        'validation': PRESENTATION_COLORS['orange'],
    }

    for split in ['train', 'validation']:
        split_history = target_history[target_history['split'] == split]
        fig.add_trace(
            go.Scatter(
                x=split_history['epoch'],
                y=split_history[metric_column],
                mode='lines+markers',
                name=split_labels[split],
                line={'color': split_colors[split], 'width': 2.6},
                marker={'size': 7, 'line': {'color': '#ffffff', 'width': 1}},
                showlegend=show_legend,
                hovertemplate=f'Epoch %{{x}}<br>{metric_label.title()}: %{{y:.3f}}<extra></extra>',
            ),
            row=row,
            col=col,
        )

        chosen_point = split_history[split_history['epoch']
                                     == chosen_epoch].iloc[0]
        fig.add_trace(
            go.Scatter(
                x=[chosen_epoch],
                y=[float(chosen_point[metric_column])],
                mode='markers',
                name='Chosen epoch',
                marker={
                    'size': 16,
                    'symbol': 'diamond',
                    'color': split_colors[split],
                    'line': {'color': '#ffffff', 'width': 2},
                },
                showlegend=(show_legend and split == 'validation'),
                hovertemplate=f'Chosen epoch %{{x}}<br>{metric_label.title()}: %{{y:.3f}}<extra></extra>',
            ),
            row=row,
            col=col,
        )


def add_chosen_epoch_annotation(
    fig: go.Figure,
    *,
    chosen_epoch: int,
    axis_range: list[float],
    row: int,
    col: int,
) -> None:
    fig.add_vline(
        x=chosen_epoch,
        line_width=1.8,
        line_dash='dash',
        line_color=PRESENTATION_COLORS['slate'],
        row=row,
        col=col,
    )
    fig.add_annotation(
        x=chosen_epoch,
        y=axis_range[1] - ((axis_range[1] - axis_range[0]) * 0.06),
        xref=f'x{col if col > 1 else ""}',
        yref=f'y{col if col > 1 else ""}',
        showarrow=False,
        yanchor='top',
        bgcolor='rgba(255, 255, 255, 0.92)',
        bordercolor='rgba(148, 163, 184, 0.45)',
        borderpad=3,
        font={'size': 12, 'color': '#475569'},
        text=f'Chosen epoch: {chosen_epoch}',
    )


def build_validation_metrics_figure(
    epoch_history: pd.DataFrame,
    chosen_epoch_summary: pd.DataFrame,
) -> go.Figure:
    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=[MODEL_LABEL_MAP['churn'],
                        MODEL_LABEL_MAP['re_engage']],
        horizontal_spacing=0.12,
    )

    churn_range = [0.45, 0.90]
    engagement_range = [0.70, 0.98]

    for col_idx, (target_key, axis_range) in enumerate(
        [('churn', churn_range), ('re_engage', engagement_range)],
        start=1,
    ):
        target_history = epoch_history[epoch_history['target'] == target_key]
        chosen_row = chosen_epoch_summary[chosen_epoch_summary['target']
                                          == target_key].iloc[0]
        chosen_epoch = int(chosen_row['chosen_epoch'])

        add_epoch_metric_traces(
            fig,
            target_history,
            chosen_epoch=chosen_epoch,
            metric_column='accuracy',
            metric_label='accuracy',
            row=1,
            col=col_idx,
            show_legend=(col_idx == 1),
        )
        add_chosen_epoch_annotation(
            fig,
            chosen_epoch=chosen_epoch,
            axis_range=axis_range,
            row=1,
            col=col_idx,
        )

    apply_presentation_layout(
        fig,
        xaxis_title='Epoch',
        yaxis_title='Accuracy',
        **FIGURE_DIMENSIONS['validation_history'],
    )
    fig.update_layout(
        legend={
            'orientation': 'h',
            'yanchor': 'bottom',
            'y': 1.08,
            'xanchor': 'left',
            'x': 0.0,
            'title': {'text': ''},
        }
    )
    fig.update_xaxes(dtick=4)
    fig.update_yaxes(range=churn_range, tickformat='.0%', row=1, col=1)
    fig.update_yaxes(range=engagement_range, tickformat='.0%', row=1, col=2)
    return fig


def build_model_family_figure(model_family_comparison: pd.DataFrame) -> go.Figure:
    metric_labels = {
        'roc_auc': 'ROC AUC',
        'average_precision': 'Avg precision',
        'brier_score': 'Brier score',
        'log_loss': 'Log loss',
        'top_decile_precision': 'Top-decile precision',
    }
    family_colors = {
        'logistic_regression': PRESENTATION_COLORS['blue'],
        'random_forest': PRESENTATION_COLORS['orange'],
    }
    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=[MODEL_LABEL_MAP['churn'],
                        MODEL_LABEL_MAP['re_engage']],
        horizontal_spacing=0.12,
    )

    for col_idx, target_key in enumerate(['churn', 're_engage'], start=1):
        target_slice = model_family_comparison[model_family_comparison['target'] == target_key]
        for family in ['logistic_regression', 'random_forest']:
            family_slice = target_slice[target_slice['model_family'] == family]
            fig.add_trace(
                go.Bar(
                    x=[metric_labels[key] for key in metric_labels],
                    y=[float(family_slice.iloc[0][key])
                       for key in metric_labels],
                    name=MODEL_FAMILY_LABELS[family],
                    marker_color=family_colors[family],
                    showlegend=(col_idx == 1),
                    hovertemplate='%{x}<br>Score: %{y:.3f}<extra></extra>',
                ),
                row=1,
                col=col_idx,
            )

    apply_presentation_layout(
        fig,
        xaxis_title='Validation metric',
        yaxis_title='Score',
        **FIGURE_DIMENSIONS['model_family_check'],
    )
    fig.update_layout(
        barmode='group',
        legend={
            'orientation': 'h',
            'yanchor': 'bottom',
            'y': 1.08,
            'xanchor': 'left',
            'x': 0.0,
            'title': {'text': ''},
        },
    )
    fig.update_yaxes(range=[0, 1.05])
    return fig


def build_calibration_figure(
    churn_result: dict[str, Any],
    engagement_result: dict[str, Any],
) -> go.Figure:
    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=[MODEL_LABEL_MAP['churn'],
                        MODEL_LABEL_MAP['re_engage']],
        horizontal_spacing=0.12,
    )
    ordered_deciles = [str(idx) for idx in range(1, 11)]

    for col_idx, (target_key, result) in enumerate(
        [
            ('churn', churn_result),
            ('re_engage', engagement_result),
        ],
        start=1,
    ):
        calibration = result['calibration'].copy()
        calibration['score_decile'] = pd.Categorical(
            calibration['score_decile'].astype(int).astype(str),
            categories=ordered_deciles,
            ordered=True,
        )
        calibration = calibration.sort_values('score_decile')

        fig.add_trace(
            go.Scatter(
                x=calibration['score_decile'].astype(str),
                y=calibration['avg_prediction'],
                mode='lines+markers',
                name='Average prediction',
                line={
                    'color': PRESENTATION_COLORS['green'], 'width': 2.6, 'dash': 'dash'},
                marker={'size': 8},
                showlegend=(col_idx == 1),
                hovertemplate='Decile %{x}<br>Average prediction: %{y:.3f}<extra></extra>',
            ),
            row=1,
            col=col_idx,
        )
        fig.add_trace(
            go.Scatter(
                x=calibration['score_decile'].astype(str),
                y=calibration['actual_rate'],
                mode='lines+markers',
                name='Actual rate',
                line={'color': PRESENTATION_COLORS['orange'], 'width': 3.0},
                marker={'size': 8, 'line': {'color': '#ffffff', 'width': 1.2}},
                showlegend=(col_idx == 1),
                hovertemplate='Decile %{x}<br>Actual rate: %{y:.3f}<extra></extra>',
            ),
            row=1,
            col=col_idx,
        )

        calibration_gap = (
            calibration['actual_rate'] - calibration['avg_prediction']).abs().max()
        fig.add_annotation(
            x=1.0,
            y=-0.22,
            xref=f'x{col_idx if col_idx > 1 else ""} domain',
            yref='paper',
            xanchor='right',
            showarrow=False,
            font={'size': 13, 'color': '#475569'},
            text=f'Max decile gap: {float(calibration_gap):.1%}',
        )
        fig.update_xaxes(
            categoryorder='array',
            categoryarray=ordered_deciles,
            row=1,
            col=col_idx,
        )

    apply_presentation_layout(
        fig,
        xaxis_title='Score decile',
        yaxis_title='Probability',
        **FIGURE_DIMENSIONS['calibration_distribution'],
    )
    fig.update_layout(
        legend={
            'orientation': 'h',
            'yanchor': 'bottom',
            'y': 1.08,
            'xanchor': 'left',
            'x': 0.0,
            'title': {'text': ''},
        },
    )
    fig.update_yaxes(range=[0, 1], tickformat='.0%')
    return fig


def build_segment_distribution_figure(segment_distribution: pd.DataFrame) -> go.Figure:
    plot_frame = segment_distribution.copy().sort_values(
        'segment_order').reset_index(drop=True)
    plot_frame['segment_label'] = plot_frame['segment_id'] + \
        ' · ' + plot_frame['segment_name']
    segment_palette = {
        'S1': '#dbeafe',
        'S2': '#bfdbfe',
        'S3': '#60a5fa',
        'S4': '#2563eb',
        'S5': '#1e3a8a',
    }

    fig = go.Figure()
    for _, row in plot_frame.iterrows():
        share_pct = float(row['share_pct'])
        users = int(row['users'])
        text_font_color = '#ffffff' if row['segment_id'] in {
            'S4', 'S5'} else '#1e3a8a'
        label_text = f"{share_pct:.1f}%" if share_pct < 8 else f"{share_pct:.1f}%<br>({users:,})"
        fig.add_trace(
            go.Bar(
                x=[share_pct],
                y=['CRM base'],
                orientation='h',
                name=str(row['segment_label']),
                marker={
                    'color': segment_palette.get(str(row['segment_id']), '#94a3b8'),
                    'line': {'color': '#ffffff', 'width': 1.2},
                },
                text=[label_text],
                textposition='inside',
                insidetextanchor='middle',
                textangle=90 if row['segment_id'] == 'S2' else 0,
                textfont={'color': text_font_color, 'size': 12},
                customdata=[[row['segment_description'],
                             users, str(row['segment_label'])]],
                hovertemplate=(
                    '<b>%{customdata[2]}</b><br>'
                    'Share: %{x:.2f}%<br>'
                    'Users: %{customdata[1]:,}<br>'
                    '%{customdata[0]}<extra></extra>'
                ),
            )
        )

    apply_presentation_layout(
        fig,
        xaxis_title='Share of latest CRM snapshot (%)',
        yaxis_title='',
        **FIGURE_DIMENSIONS['crm_segment_distribution'],
    )
    fig.update_layout(
        barmode='stack',
        margin={'l': 20, 'r': 20, 't': 70, 'b': 70},
        legend={
            'orientation': 'h',
            'yanchor': 'bottom',
            'y': 1.04,
            'xanchor': 'left',
            'x': 0.0,
            'title': {'text': ''},
            'font': {'size': 13},
        },
    )
    fig.update_xaxes(range=[0, 100], ticksuffix='%', dtick=10)
    fig.update_yaxes(showticklabels=False, showgrid=False)

    return fig

## Validation Metrics Output

This section summarizes model quality on the temporal holdout. The summary table reports overall validation performance, while the epoch chart shows whether the training process is stable across passes through the data.

The two targets behave differently. Re-engagement is the stronger task on every headline metric: `ROC AUC = 0.9635`, `average precision = 0.9229`, and `top-decile precision = 0.9582`. Churn is still useful, but clearly less stable: `ROC AUC = 0.7981`, `average precision = 0.8763`, and `top-decile precision = 0.9524`. In practice, both models identify the highest-priority users well, but churn is weaker once the full score range is considered.

The epoch view supports the same conclusion. Churn fluctuates more and reaches its best validation point early, so epoch `3` is a reasonable stopping point. Re-engagement stabilizes quickly and then remains flat, which makes epoch `14` acceptable without being uniquely important. Validation accuracy being above training accuracy for re-engagement is notable, but it does not by itself indicate leakage: the split remains temporal and the ROC AUC profile is stable across epochs.

![Validation Metrics Output](images/09-validation-metric-output.svg)

In [20]:
churn_results: dict[str, Any] = fit_propensity_model(
    churn_frame,
    label_column='churn_30_to_60',
    score_column='churn_30_to_60_prob',
    feature_columns=shared_features,
    categorical_columns=categorical_features,
)

engagement_results: dict[str, Any] = fit_propensity_model(
    engagement_frame,
    label_column='re_engage_30d',
    score_column='re_engage_30d_prob',
    feature_columns=shared_features,
    categorical_columns=categorical_features,
)

validation_metrics = pd.DataFrame(
    [
        {'model': 'churn', **churn_results['validation_metrics'].to_dict()},
        {'model': 're_engage', **
            engagement_results['validation_metrics'].to_dict()},
    ]
).round(4)

validation_epoch_history = pd.concat(
    [
        build_epoch_metric_history(
            churn_frame,
            label_column='churn_30_to_60',
            feature_columns=shared_features,
            categorical_columns=categorical_features,
            target_label='churn',
        ),
        build_epoch_metric_history(
            engagement_frame,
            label_column='re_engage_30d',
            feature_columns=shared_features,
            categorical_columns=categorical_features,
            target_label='re_engage',
        ),
    ],
    ignore_index=True,
)

validation_epoch_table, chosen_epoch_summary = select_epoch_history(
    validation_epoch_history,
)
validation_epoch_table_display = validation_epoch_table.assign(
    target=lambda df: df['target'].map(MODEL_LABEL_MAP),
    train_accuracy=lambda df: df['train_accuracy'].map(
        lambda value: f'{float(value):.2%}'),
    validation_accuracy=lambda df: df['validation_accuracy'].map(
        lambda value: f'{float(value):.2%}'),
    train_roc_auc=lambda df: df['train_roc_auc'].map(
        lambda value: f'{float(value):.3f}'),
    validation_roc_auc=lambda df: df['validation_roc_auc'].map(
        lambda value: f'{float(value):.3f}'),
    chosen_epoch=lambda df: np.where(df['chosen_epoch'], 'yes', ''),
)[[
    'target',
    'epoch',
    'train_accuracy',
    'validation_accuracy',
    'train_roc_auc',
    'validation_roc_auc',
    'chosen_epoch',
]]
churn_epoch_table_display = validation_epoch_table_display[
    validation_epoch_table_display['target'] == MODEL_LABEL_MAP['churn']
].reset_index(drop=True)
engagement_epoch_table_display = validation_epoch_table_display[
    validation_epoch_table_display['target'] == MODEL_LABEL_MAP['re_engage']
].reset_index(drop=True)

fig_validation_metrics = build_validation_metrics_figure(
    validation_epoch_history,
    chosen_epoch_summary,
)
fig_validation_metrics.show()

print('Churn epoch accuracy and ROC AUC table')
display(churn_epoch_table_display)
print('Re-engagement epoch accuracy and ROC AUC table')
display(engagement_epoch_table_display)

validation_metrics

Churn epoch accuracy and ROC AUC table


split,target,epoch,train_accuracy,validation_accuracy,train_roc_auc,validation_roc_auc,chosen_epoch
0,Churn,1,62.81%,62.35%,0.671,0.723,
1,Churn,2,64.90%,58.59%,0.677,0.719,
2,Churn,3,69.03%,69.54%,0.734,0.799,yes
3,Churn,4,55.95%,46.52%,0.656,0.614,
4,Churn,5,68.17%,63.79%,0.727,0.794,
5,Churn,6,66.60%,66.75%,0.725,0.758,
6,Churn,7,66.42%,63.87%,0.703,0.701,
7,Churn,8,66.05%,61.31%,0.693,0.704,
8,Churn,9,67.42%,61.71%,0.747,0.747,
9,Churn,10,65.21%,63.95%,0.701,0.736,


Re-engagement epoch accuracy and ROC AUC table


split,target,epoch,train_accuracy,validation_accuracy,train_roc_auc,validation_roc_auc,chosen_epoch
0,Re-engagement,1,83.21%,88.28%,0.904,0.948,
1,Re-engagement,2,85.36%,90.08%,0.923,0.959,
2,Re-engagement,3,84.20%,89.94%,0.918,0.959,
3,Re-engagement,4,84.66%,90.15%,0.923,0.963,
4,Re-engagement,5,85.38%,90.50%,0.925,0.963,
5,Re-engagement,6,85.10%,90.63%,0.924,0.964,
6,Re-engagement,7,85.18%,90.57%,0.926,0.963,
7,Re-engagement,8,85.27%,90.30%,0.924,0.961,
8,Re-engagement,9,84.58%,90.00%,0.921,0.958,
9,Re-engagement,10,84.95%,90.45%,0.925,0.962,


,model,rows,positive_rate,roc_auc,average_precision,brier_score,log_loss,top_decile_precision
0,churn,1251.0,0.6563,0.7981,0.8763,0.2188,0.6288,0.9524
1,re_engage,19115.0,0.3813,0.9635,0.9229,0.0709,0.2382,0.9582


## Model Family Check

The random forest performs better, but the improvement is moderate rather than transformative. On churn it increases `ROC AUC` from `0.7981` to `0.8062`, and on re-engagement from `0.9635` to `0.9690`. It also improves `top_decile_precision` for both tasks.

That tradeoff is important in this project. Logistic regression remains easier to explain, calibrate, and connect to business rules, so it stays the default path for exported scores and CRM decisions. The random forest is still useful as a benchmark, because it shows that the simpler baseline is close to the stronger nonlinear alternative.

![Model Family Check](images/10-model-family-check.svg)

In [21]:
rf_churn_results: dict[str, Any] = fit_propensity_model(
    churn_frame,
    label_column='churn_30_to_60',
    score_column='churn_30_to_60_rf_prob',
    feature_columns=shared_features,
    categorical_columns=categorical_features,
    pipeline_builder=build_random_forest_pipeline,
    model_family='random_forest',
)

rf_engagement_results: dict[str, Any] = fit_propensity_model(
    engagement_frame,
    label_column='re_engage_30d',
    score_column='re_engage_30d_rf_prob',
    feature_columns=shared_features,
    categorical_columns=categorical_features,
    pipeline_builder=build_random_forest_pipeline,
    model_family='random_forest',
)

model_family_comparison = pd.DataFrame(
    [
        {'target': 'churn', 'model_family': 'logistic_regression',
            **churn_results['validation_metrics'].to_dict()},
        {'target': 'churn', 'model_family': 'random_forest', **
            rf_churn_results['validation_metrics'].to_dict()},
        {'target': 're_engage', 'model_family': 'logistic_regression',
            **engagement_results['validation_metrics'].to_dict()},
        {'target': 're_engage', 'model_family': 'random_forest', **
            rf_engagement_results['validation_metrics'].to_dict()},
    ]
).round(4)

fig_model_family = build_model_family_figure(model_family_comparison)
fig_model_family.show()
model_family_comparison

,target,model_family,rows,positive_rate,roc_auc,average_precision,brier_score,log_loss,top_decile_precision
0,churn,logistic_regression,1251.0,0.6563,0.7981,0.8763,0.2188,0.6288,0.9524
1,churn,random_forest,1251.0,0.6563,0.8062,0.8804,0.2096,0.6042,0.9603
2,re_engage,logistic_regression,19115.0,0.3813,0.9635,0.9229,0.0709,0.2382,0.9582
3,re_engage,random_forest,19115.0,0.3813,0.9690,0.9395,0.0679,0.2215,0.9759


## Feature Relevance Check

These tables provide the interpretability layer. They do not imply causality, but they do show whether the models rely on signals that are coherent from a business perspective.

For churn, the dominant signals are mainly point status and recent context: `platform_missing`, `points_earned_lifetime_proxy`, `points_redeemed_lifetime`, `totalPoints`, and several month effects. The random forest points in a similar direction, with `points_gap_proxy`, `totalPoints`, and reward-eligibility variables near the top. Overall, the churn score appears to depend more on accumulated value, distance from reward thresholds, and some time-specific variation than on a single behavioral trigger.

Re-engagement is more concentrated around recent behavior. `days_since_last_activity` is the strongest logistic signal, followed by `days_since_last_login`, recent mission activity, and point variables. The random forest places even more weight on recent scans and earned points.

In [22]:
def format_feature_relevance(result: dict[str, Any], target: str) -> pd.DataFrame:
    relevance = result['feature_relevance'].copy()
    relevance.insert(0, 'target', target)
    relevance.insert(1, 'model_family', result['model_family'])
    return relevance[['target', 'model_family', 'feature', 'relevance_type', 'direction', 'relevance']].round(4)


churn_feature_relevance = pd.concat(
    [
        format_feature_relevance(churn_results, 'churn'),
        format_feature_relevance(rf_churn_results, 'churn'),
    ],
    ignore_index=True,
)

engagement_feature_relevance = pd.concat(
    [
        format_feature_relevance(engagement_results, 're_engage'),
        format_feature_relevance(rf_engagement_results, 're_engage'),
    ],
    ignore_index=True,
)
print('Churn feature relevance')
display(churn_feature_relevance)
print('Re-engagement feature relevance')
display(engagement_feature_relevance)

Churn feature relevance


,target,model_family,feature,relevance_type,direction,relevance
0,churn,logistic_regression,platform_missing,coefficient,raises score,1.6318
1,churn,logistic_regression,points_earned_lifetime_proxy,coefficient,lowers score,-1.0794
2,churn,logistic_regression,reference_month_2025-06,coefficient,raises score,0.6463
3,churn,logistic_regression,reference_month_2025-01,coefficient,lowers score,-0.6246
4,churn,logistic_regression,points_redeemed_lifetime,coefficient,raises score,0.5692
5,churn,logistic_regression,platform_android,coefficient,lowers score,-0.5675
6,churn,logistic_regression,platform_ios,coefficient,lowers score,-0.5146
7,churn,logistic_regression,totalPoints,coefficient,raises score,0.4997
8,churn,logistic_regression,points_gap_proxy,coefficient,raises score,0.3354
9,churn,logistic_regression,reference_month_2024-07,coefficient,raises score,0.3347


Re-engagement feature relevance


,target,model_family,feature,relevance_type,direction,relevance
0,re_engage,logistic_regression,days_since_last_activity,coefficient,lowers score,-4.9240
1,re_engage,logistic_regression,days_since_last_login,coefficient,raises score,3.1289
2,re_engage,logistic_regression,points_earned_lifetime_proxy,coefficient,raises score,1.6240
3,re_engage,logistic_regression,points_redeemed_lifetime,coefficient,lowers score,-1.2039
4,re_engage,logistic_regression,platform_missing,coefficient,lowers score,-1.1342
5,re_engage,logistic_regression,reference_month_2024-10,coefficient,lowers score,-1.1305
6,re_engage,logistic_regression,mission_count_60d,coefficient,raises score,0.7966
7,re_engage,logistic_regression,totalPoints,coefficient,lowers score,-0.6398
8,re_engage,logistic_regression,missions_completed_60d,coefficient,lowers score,-0.5685
9,re_engage,logistic_regression,child_age_bucket_pregnancy,coefficient,lowers score,-0.4796


## Churn and Re-engagement Calibration

This section checks whether the predicted probabilities behave like probabilities, not only like rankings. The chart gives the overall pattern, while the decile tables provide the exact holdout values behind it.

Re-engagement is reasonably well calibrated. The lower deciles are close to zero-risk, the upper deciles are high in the expected direction, and the gap between average prediction and actual rate remains fairly contained except around the middle buckets. The largest mismatch appears around decile `6`, where the model predicts `42.1%` and the holdout realizes `29.5%`.

Churn is less well calibrated. The ranking remains useful, but the actual rate is above the average prediction in almost every decile. In other words, the churn model separates users reasonably well, but it is conservative as a probability model. That is still workable for campaign prioritization, although the raw churn percentages should be interpreted more carefully than the re-engagement probabilities.

![Churn and Re-engagement Calibration](images/11-churn-re-engagement-calibration.svg)


In [23]:
fig_calibration = build_calibration_figure(churn_results, engagement_results)
fig_calibration.show()

print('Churn calibration')
display(churn_results['calibration'])

print('Re-engagement calibration')
display(engagement_results['calibration'])

Churn calibration


,score_decile,rows,avg_prediction,actual_rate
0,1,126,0.131421,0.285714
1,2,125,0.202870,0.336000
2,3,125,0.253262,0.408000
3,4,125,0.308315,0.536000
4,5,125,0.366578,0.696000
5,6,125,0.434623,0.736000
6,7,125,0.523822,0.800000
7,8,125,0.654059,0.944000
8,9,125,0.758254,0.872000
9,10,125,0.863484,0.952000


Re-engagement calibration


,score_decile,rows,avg_prediction,actual_rate
0,1,1912,0.000109,0.000000
1,2,1911,0.000650,0.003663
2,3,1912,0.004336,0.003661
3,4,1911,0.021294,0.010989
4,5,1912,0.105543,0.053870
5,6,1911,0.420628,0.294610
6,7,1911,0.718002,0.692831
7,8,1912,0.852318,0.862448
8,9,1911,0.925610,0.933019
9,10,1912,0.973679,0.958159


## CRM Segment Distribution

This is the final translation step from model outputs to operational action buckets. The chart gives the aggregate volume view, while the table underneath keeps the exact segment definitions and counts visible.

The latest scored base is concentrated in three segments. `S5 - Stable passive, points far` is the largest group at `47.0%` (`2,919` users), `S3 - Stable user, points close` follows at `24.9%` (`1,547`), and `S1 - Physiological churn` remains sizeable at `20.4%` (`1,265`). `S2` and `S4` are much smaller at `2.3%` and `5.3%`.

From a CRM perspective, this distribution is useful because it keeps most of the audience in a few broad operational buckets while still preserving the smaller edge cases as distinct segments.

![CRM Segment Distribution](images/12-crm-segment-distribution.svg)

In [24]:
crm_engine = CRMDecisionEngine(
    project_root=Path('.').resolve(),
    artifacts_dir=Path('artifacts').resolve(),
    rules_path=Path('decision_rules.txt').resolve(),
)

segment_reference = pd.DataFrame(
    [
        {
            'segment_id': rule['segment_id'],
            'segment_name': rule['segment_name'],
            'segment_description': rule['segment_description'],
        }
        for rule in crm_engine.rules
    ]
).drop_duplicates()
segment_reference['segment_order'] = segment_reference['segment_id'].str.extract(
    r'(\d+)').astype(int)

segment_counts = (
    pd.Series(crm_engine.segment_lookup, name='segment_id')
    .value_counts()
    .rename_axis('segment_id')
    .reset_index(name='users')
)

segment_distribution = (
    segment_reference.merge(segment_counts, on='segment_id', how='left')
    .fillna({'users': 0})
    .assign(users=lambda df: df['users'].astype(int))
    .assign(share_pct=lambda df: (100 * df['users'] / df['users'].sum()).round(2))
    .sort_values(['segment_order', 'segment_id'])
    .reset_index(drop=True)
)

fig_segment_distribution = build_segment_distribution_figure(
    segment_distribution)
fig_segment_distribution.show()

segment_distribution[[
    'segment_id',
    'segment_name',
    'segment_description',
    'users',
    'share_pct',
]]

,segment_id,segment_name,segment_description,users,share_pct
0,S1,Physiological churn,Users in the lifecycle transition window who s...,1265,20.39
1,S2,At-risk rescue,High-churn users who still fit the app lifecyc...,143,2.30
2,S3,"Stable user, points close",Low-churn users who are already near a reward ...,1547,24.93
3,S4,"Stable active, points far",Low-churn engaged users who are far from a rew...,331,5.33
4,S5,"Stable passive, points far",Users with low churn risk but weak current mom...,2919,47.04
